# 07 — Tearsheets & Final Comparison
Pull everything from the results store: rank all runs, produce quantstats
tearsheets for the finalists, and break their performance down by regime.

### Results store ranking
All runs from chapters 03–05. Sort by Sharpe, pick best.


In [ ]:
import sys
sys.path.insert(0, "../src")
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.width", 160)

### Re-run best + tearsheet
Quantstats HTML: Sharpe, Sortino, max DD, monthly rets, distributions.


In [ ]:
from lab.backtest import StrategyConfig, run_backtest
from lab.experiments import load_runs, load_trades
from lab.report import tearsheet, compare_runs, regime_breakdown

all_runs = load_runs()
print(f"{len(all_runs)} runs in the store, tags: {sorted(all_runs.tag.unique())}")
compare_runs().round(3).head(15)

### Regime breakdown
VIX quartile performance. Vol-dependent or robust?


In [ ]:
# Re-run the best config end-to-end and write its HTML tearsheet
best = all_runs.sort_values("sharpe_ratio", ascending=False).iloc[0]
print("best run:", best["name"], best.config_hash)
trades = load_trades(best.config_hash)
trades.tail()

### Final equity curve
Cumulative return over time. Smooth = good. Drawdowns = when/why?


In [ ]:
import json
cfg = StrategyConfig(
    name=best["name"], strategy=best["strategy"],
    params=json.loads(best["params_json"]),
    entry_filter=best["entry_filter"] or None,
    start=best["start"] or None, end=best["end"] or None,
)
result = run_backtest(cfg)
path = tearsheet(result)
print("tearsheet ->", path)

In [ ]:
regime_breakdown(result.trade_log, feature="vix_rank", bins=4).round(3)

In [ ]:
result.equity_curve.plot(title=f"{cfg.name} — final equity curve");